In [69]:
import plotly
import plotly.graph_objects as go
import plotly.express as px
import numpy as np
import pandas as pd
from scipy.signal import find_peaks

import utils_for_plotly_rewrites

In [2]:
lambda_tokens = ["nm", "wavelength", "wavelength_nm", "lambda", "lambda_nm", "wl", "wl_nm", "Observed", "Observed Wavelength", "obs", "wave"]

int_tokens =["Grey Val", "grey val", "gray val", "grayscale", "gray value", "intensity", "signal", "counts", "value", "int", "rel. int.", "grey", "Rel. Int.", "Relative Intensity", "Rel Int", "Intensity", "A", "Aki", "gA", "gf", "weighted f", "f", "Intensity/Counts", 'rel', 'count', 'flux', 'grey value',]

# Shared Values
Y_TITLE = 'Intensity'
X_TITLE = 'Wavelength (nm)'
COLOUR = 'white'
BG = 'black'
X_MIN = 400
X_MAX = 750
FIG_WIDTH = 15
DPI = 600

# Traditional Plot Values
FIG_HEIGHT_BASE = 3.0
FIG_SIZE = (FIG_WIDTH, FIG_HEIGHT_BASE)
MIN_NEEDLE_WIDTH = 0.1
MAX_NEEDLE_WIDTH = 0.3
MAX_Y_SCALE = 0.75
NEEDLE_POWER_SHAPE = 4
LABEL_NORM_INT = 0.20
GLOW_WIDTH_MULT = 1.3

# Dynamic Height Values (overflow section)
fig_height_overflow_scale = 9.0

# Normalised-Specific Values
NORM_PROM_PERC = 0.15
NORM_MIN_BRIGHT = 0.01
NORM_GLOW_ALPHA = 0
NORM_PEAK_EMPHASIS = 1.1
NORM_PEAK_LABEL_POSN = 0.75

# "Default" Values (i.e. for not normalised)
DEFAULT_PROM_PERC = 0.08    # also used by "other" plots
DEFAULT_MIN_BRIGHT = 0.1    # also used by "other" plots
DEFAULT_GLOW_ALPHA = 0.35
DEFAULT_PEAK_EMPHASIS = 1.4
DEFAULT_PEAK_LABEL_POSN = 0.77

# Other Plot Values
fig_size = (15,6)
#prominence = 0.08       # changed from 0.12
#min_bright = 0.1
min_alpha = 0.1
min_alpha_scatter = 0.2
base_marker_size = 5
max_marker_size_factor = 95
gamma_factor = 0.8
bar_width = 1
smoothing_window = 5    # Increase this value to control the degree of smoothing
base_sigma_nm = 0.5 # Base width of the Gaussian (for low intensity peaks)
max_sigma_multiplier = 4.0 # How much wider the highest intensity peaks can be
reverse_x = True
plot_type = None
show_grid = True

In [4]:
HE_SHEET = "he test.xlsx"

# Load the data directly from the Google Sheet URL
df = pd.read_excel(HE_SHEET)

df_filtered = df[(df['nm'] >= 400) & (df['nm'] <= 750)].copy()

# Sort by 'nm' to ensure correct neighbor comparison and plotting order
df_filtered = df_filtered.sort_values(by='nm').reset_index(drop=True)

print("DataFrame loaded successfully from Google Sheet. Here are the first 5 rows:")
print(df.head())

min_grey_val = df_filtered['Grey Val'].min()
max_grey_val = df_filtered['Grey Val'].max()
grey_val_range = max_grey_val - min_grey_val

# Filter the DataFrame for 'nm' values between 400 and 750
df_filtered = df[(df['nm'] >= 400) & (df['nm'] <= 750)].copy()

# Sort by 'nm' to ensure correct neighbor comparison and plotting order
df_filtered = df_filtered.sort_values(by='nm').reset_index(drop=True)


DataFrame loaded successfully from Google Sheet. Here are the first 5 rows:
   Unnamed: 0  Grey Val         nm  Unnamed: 3    Unnamed: 4  Unnamed: 5  \
0        2178   247.148  589.03701         NaN  max grey val         NaN   
1        2179   246.463  587.91075         NaN       247.148         NaN   
2        2180   243.830  586.78449         NaN           NaN         NaN   
3        2181   241.833  585.65823         NaN  min grey val         NaN   
4        2177   240.733  590.16327         NaN        16.674         NaN   

   Unnamed: 6 Unnamed: 7  
0  3042.03129   -1.12626  
1         NaN        NaN  
2         NaN        NaN  
3         NaN        NaN  
4         NaN    he_3584  


In [80]:
df_filtered = df[(df['nm'] >= 400) & (df['nm'] <= 750)].copy()
df_filtered = df_filtered.sort_values(by='nm').reset_index(drop=True)


min_int = df_filtered['Grey Val'].min()
max_int = df_filtered['Grey Val'].max()

int_range = max_int - min_int

if int_range == 0:
    df_filtered['Normalized_int'] = 1.0
else:

    df_filtered['Normalized_int'] = (df_filtered['Grey Val'] - min_int) / int_range

In [73]:
# --- Prepare Data for Plotting ---
x_wavelengths = df_filtered['nm'].values # Wavelengths
z_intensity = df_filtered['Grey Val'].values # Grey Values (Intensity)
spectral_colors = [utils_for_plotly_rewrites.rgb(nm) for nm in x_wavelengths]

# --- Plotly 2D Full-Color Emission Spectrum Plot ---
fig_2d = go.Figure()

for i in range(len(x_wavelengths) - 1):
    fig_2d.add_trace(go.Scatter(
        x=x_wavelengths[i:i+2],
        y=z_intensity[i:i+2],
        mode='lines',
        line={"color": f'rgb({int(spectral_colors[i][0]*255)}, {int(spectral_colors[i][1]*255)}, {int(spectral_colors[i][2]*255)})', "width": 3},
        showlegend=False,
        hoverinfo='x+y'
    ))

fig_2d.update_layout(
    title_text='2D Full-Colour Emission Spectrum (Plotly Line Plot, Dark Mode)',
    title_x=0.5,
    xaxis_title='Wavelength (nm)',
    yaxis_title='Grey Value (Intensity)',
    template='plotly_dark',
    plot_bgcolor='black',
    paper_bgcolor='black',
    font={"color": 'white'},
    xaxis={
        "gridcolor": 'darkgray',
        "zerolinecolor": 'darkgray',
        "showline": True, "linecolor": 'darkgray'
    },
    yaxis={
        "gridcolor": 'darkgray',
        "zerolinecolor": 'darkgray',
        "showline": True, "linecolor": 'darkgray'
    }
)

fig_2d.update_xaxes(range=[x_wavelengths.min(), x_wavelengths.max()])
fig_2d.update_yaxes(range=[0, z_intensity.max() * 1.1])

fig_2d.show()

In [75]:
# --- Prepare Data for Plotting ---
x_wavelengths = df_filtered['nm'].values 
z_intensity = df_filtered['Grey Val'].values 
spectral_colors = [utils_for_plotly_rewrites.rgb(nm) for nm in x_wavelengths]

img_width = len(x_wavelengths)
img_height = 256 

spectrum_image = np.zeros((img_height, img_width, 3), dtype=np.uint8)
max_intensity = z_intensity.max() if z_intensity.max() > 0 else 1

# --- Loop with Explicit Tuple Indexing Fixed ---
for i in range(img_width):
    current_intensity = z_intensity[i]
    color_normalized = spectral_colors[i]
    
    # Pulled directly from your logic with specific index items [0], [1], [2]
    color_rgb_int = (
        int(color_normalized[0] * 255), 
        int(color_normalized[1] * 255), 
        int(color_normalized[2] * 255)
    )
    
    num_pixels_to_fill = int((current_intensity / max_intensity) * img_height)
    for j in range(img_height - num_pixels_to_fill, img_height):
        spectrum_image[j, i] = color_rgb_int

# --- Calculate Half-Pixel Coordinate Adjustments ---
min_x, max_x = x_wavelengths.min(), x_wavelengths.max()
min_y, max_y = 0.0, float(max_intensity)

# Determine structural stride step size per pixel block
dx = (max_x - min_x) / img_width
dy = (max_y - min_y) / img_height

# Shift x0 and y0 to pixel centers so outer pixel borders clip flush against data limits
x0_centered = min_x + (dx / 2)
y0_centered = max_y - (dy / 2)

# Pass the clean dimensions directly to go.Image
fig_image_mapped = go.Figure(go.Image(
    z=spectrum_image,
    x0=x0_centered,
    dx=dx,
    y0=y0_centered,
    dy=-dy
))

# --- Layout Configuration ---
fig_image_mapped.update_layout(
    title_text='RGB Emission Spectrum (Plotly Intensity-Mapped Image, Dark Mode)',
    title_x=0.5,
    xaxis_title='Wavelength (nm)',
    yaxis_title='Intensity (Grey Value)',
    template='plotly_dark',
    plot_bgcolor='black',
    paper_bgcolor='black',
    font={"color": 'white'},
    xaxis={
        "gridcolor": 'gray',
        "zerolinecolor": 'gray',
        "showline": True,
        "linecolor": 'white',
        "range": [min_x, max_x],
        "constrain": 'domain'
    },
    yaxis={
        "gridcolor": 'gray',
        "zerolinecolor": 'gray',
        "showline": True,
        "linecolor": 'white',
        "range": [0, max_y * 1.1],
        "tickmode": 'array',
        "tickvals": np.linspace(0, max_y, 5),
        "constrain": 'domain'
    }
)



fig_image_mapped.show()


In [76]:


# --- Prepare Data for Plotting ---
x_wavelengths = df_filtered['nm'].values # Wavelengths
z_intensity = df_filtered['Grey Val'].values # Grey Values (Intensity)
spectral_colors = [utils_for_plotly_rewrites.rgb(nm) for nm in x_wavelengths]

# Scale point size by Grey Val for visual impact, normalizing for better display
max_grey_val = z_intensity.max()
# Adding a small constant to avoid zero size for zero Grey Val and scaling for visibility
sizes = (z_intensity / max_grey_val) * 10 + 5 # Scale to a visible range, add a base size

# --- Plotly Color-Coded Scatter Plot ---
fig_color_scatter = go.Figure()

fig_color_scatter.add_trace(go.Scatter(
    x=x_wavelengths,
    y=z_intensity,
    mode='markers',
    marker=dict(
        color=[f'rgb({int(c[0]*255)}, {int(c[1]*255)}, {int(c[2]*255)})' for c in spectral_colors],
        size=sizes, # Use calculated sizes
        line=dict(width=0)
    ),
    showlegend=False,
    hoverinfo='x+y'
))

fig_color_scatter.update_layout(
    title_text='RGB Emission Spectrum (Plotly Color-Coded Scatter Plot, Dark Mode)',
    title_x=0.5,
    xaxis_title='Wavelength (nm)',
    yaxis_title='Grey Value (Intensity)',
    #template='plotly_dark',
    plot_bgcolor='black',
    paper_bgcolor='black',
    font=dict(color='white'),
    xaxis=dict(
        gridcolor='darkgrey',
        zerolinecolor='darkgrey',
        showline=True, linecolor='darkgrey'
    ),
    yaxis=dict(
        gridcolor='darkgrey',
        zerolinecolor='darkgrey',
        showline=True, linecolor='darkgrey'
    )
)

fig_color_scatter.update_xaxes(range=[x_wavelengths.min(), x_wavelengths.max()])
fig_color_scatter.update_yaxes(range=[0, z_intensity.max() * 1.1])

fig_color_scatter.show()

In [77]:
import plotly.graph_objects as go
import numpy as np
import pandas as pd


df = pd.read_csv(HE_SHEET)

# Filter for the visible spectrum range
df_filtered = df[(df['nm'] >= 400) & (df['nm'] <= 750)].copy()
df_filtered = df_filtered.sort_values(by='nm').reset_index(drop=True)

# --- 3. Prepare Data for Plotting ---
x_wavelengths = df_filtered['nm'].values
z_intensity = df_filtered['Grey Val'].values
spectral_colors = [utils_for_plotly_rewrites.rgb(nm) for nm in x_wavelengths]

# --- 4. Plotly Bar-Based Spectrum Plot ---
fig_bar = go.Figure()

# Loop through each data point and add it as an individual colored bar
for i in range(len(x_wavelengths)):
    current_nm = x_wavelengths[i]
    current_intensity = z_intensity[i]
    color_normalized = spectral_colors[i]
    
    # Convert normalized color [0,1] to standard string format 'rgb(R,G,B)' [0,255]
    color_str = f"rgb({int(color_normalized[0]*255)}, {int(color_normalized[1]*255)}, {int(color_normalized[2]*255)})"
    
    fig_bar.add_trace(go.Bar(
        x=[current_nm],
        y=[current_intensity],
        marker_color=color_str,
        marker_line_width=0,       # Remove borders to avoid black lines between columns
        showlegend=False,
        hoverinfo='x+y'            # Keep hover labels clean
    ))

# --- 5. Styling Layout (Dark Mode) ---
# Dynamically determine step sizes for clean gaps and widths
max_intensity = z_intensity.max() if z_intensity.max() > 0 else 1

fig_bar.update_layout(
    title_text='RGB Emission Spectrum (Plotly Colored Bar Chart, Dark Mode)',
    title_x=0.5,
    xaxis_title='Wavelength (nm)',
    yaxis_title='Intensity (Grey Value)',
    template='plotly_dark',
    plot_bgcolor='black',
    paper_bgcolor='black',
    font={"color": 'white'},
    bargap=0,                      # Force bars to touch horizontally
    xaxis={
        "gridcolor": 'gray',
        "zerolinecolor": 'gray',
        "showline": True,
        "linecolor": 'white',
        "range": [x_wavelengths.min(), x_wavelengths.max()]
    },
    yaxis={
        "gridcolor": 'gray',
        "zerolinecolor": 'gray',
        "showline": True,
        "linecolor": 'white',
        "range": [0, max_intensity * 1.1],
        "tickmode": 'array',
        "tickvals": np.linspace(0, max_intensity, 5)
    }
)

fig_bar.show()


In [78]:
import plotly.graph_objects as go
import numpy as np
import pandas as pd

# --- Data Loading and Filtering (repeated for self-containment) ---
HE_SHEET = 'https://docs.google.com/spreadsheets/d/1oz8Bpbbe8lhDfOaNUL-lwlsI6Ls6hjAu/export?format=csv'
df = pd.read_csv(HE_SHEET)
df_filtered = df[(df['nm'] >= 400) & (df['nm'] <= 750)].copy()
df_filtered = df_filtered.sort_values(by='nm').reset_index(drop=True)


# --- Prepare Data for Plotting ---
x_wavelengths = df_filtered['nm'].values # Wavelengths
z_intensity = df_filtered['Grey Val'].values # Grey Values (Intensity)
spectral_colors = [utils_for_plotly_rewrites.rgb(nm) for nm in x_wavelengths]

# --- Plotly 2D Filled Area Plot ---
fig_filled_area = go.Figure()

# Plot each segment as a filled area
for i in range(len(x_wavelengths) - 1):
    color_rgb = f'rgb({int(spectral_colors[i][0]*255)}, {int(spectral_colors[i][1]*255)}, {int(spectral_colors[i][2]*255)})'
    fig_filled_area.add_trace(go.Scatter(
        x=x_wavelengths[i:i+2],
        y=z_intensity[i:i+2],
        mode='lines',
        line=dict(color=color_rgb, width=0), # Set width to 0 for a continuous fill effect
        fill='tozeroy', # Fill to the x-axis
        fillcolor=color_rgb,
        showlegend=False,
        hoverinfo='x+y'
    ))

fig_filled_area.update_layout(
    title_text='2D Full-Colour Emission Spectrum (Plotly Filled Area Plot, Dark Mode)',
    title_x=0.5,
    xaxis_title='Wavelength (nm)',
    yaxis_title='Grey Value (Intensity)',
    template='plotly_dark',
    plot_bgcolor='black',
    paper_bgcolor='black',
    font=dict(color='white'),
    xaxis=dict(
        gridcolor='gray',
        zerolinecolor='gray',
        showline=True, linecolor='white'
    ),
    yaxis=dict(
        gridcolor='gray',
        zerolinecolor='gray',
        showline=True, linecolor='white'
    )
)

fig_filled_area.update_xaxes(range=[x_wavelengths.min(), x_wavelengths.max()])
fig_filled_area.update_yaxes(range=[0, z_intensity.max() * 1.1])

fig_filled_area.show()

In [61]:
import plotly.express as px
import plotly.graph_objects as go
from scipy.signal import find_peaks

# Create the line plot using Plotly Express with a specific color sequence
fig = px.line(df_filtered, x='nm', y='Grey Val',
              title='Grey Val vs. nm (nm range 400-750) with Sharp Peak Labels',
              color_discrete_sequence=['#1f77b4']) # Matching Matplotlib's default blue

# Update layout for dark background, white text, and set figure size
fig.update_layout(
    plot_bgcolor='black',
    paper_bgcolor='black',
    font_color='white',
    title_font_color='white',
    width=960,  # Increased width for 12 inches (approx. 80px/inch)
    height=560, # Increased height for 7 inches (approx. 80px/inch)
    xaxis=dict(
        title='Wavelength (nm)',
        gridcolor='#333333', # Even darker grey for grid
        gridwidth=1,
        griddash='dot', # Make grid lines dotted
        showgrid=True,
        showline=True, # Show the axis line
        linecolor='#333333', # Even darker grey for axis line
        linewidth=1,
        mirror=True # Draw ticks and lines at opposite side of the plot to create a box
    ),
    yaxis=dict(
        title='Grey Val (Intensity)',
        gridcolor='#333333', # Even darker grey for grid
        gridwidth=1,
        griddash='dot', # Make grid lines dotted
        showgrid=True,
        showline=True, # Show the axis line
        linecolor='#333333', # Even darker grey for axis line
        linewidth=1,
        mirror=True
    )
)

# Recalculate dynamic_prominence if not already available in the environment
# This ensures the cell is self-contained if run independently later
if 'min_grey_val' not in locals() or 'max_grey_val' not in locals():
    min_grey_val = df_filtered['Grey Val'].min()
    max_grey_val = df_filtered['Grey Val'].max()
    grey_val_range = max_grey_val - min_grey_val
    prominence_percentage = 0.12
    dynamic_prominence = prominence_percentage * grey_val_range


peaks, _ = find_peaks(df_filtered['Grey Val'], prominence=dynamic_prominence)

# Add annotations for the identified sharp peaks
for peak_index in peaks:
    row = df_filtered.iloc[peak_index]
    fig.add_annotation(
        x=row['nm'],
        y=row['Grey Val'],
        text=f"{row['nm']:.2f} nm",
        showarrow=True,
        arrowhead=2,
        arrowcolor="black",
        arrowsize=1,
        arrowwidth=2,
        ax=0,
        ay=-30, # Offset for text
        bgcolor="red",
        bordercolor='#333333', # Even darker grey for annotation border
        borderwidth=0.5,
        borderpad=4,
        opacity=0.7,
        font=dict(color="black") # Text color for annotation
    )

fig.show()

In [68]:
import plotly.graph_objects as go
import plotly.express as px

# Create a Plotly figure
fig = go.Figure()

# Iterate through df_filtered to draw line segments with custom colors
for i in range(len(df_filtered) - 1):
    wavelength_start = df_filtered.iloc[i]['nm']
    wavelength_end = df_filtered.iloc[i+1]['nm']

    int_factor = df_filtered.iloc[i]['Normalized_int']
    # rgb, colored_rgb, final_scale, gamma_factor, min_bright are expected to be defined in the global scope.
    base_rgb_val = utils_for_plotly_rewrites.rgb(wavelength_start, gamma=gamma_factor)
    final_intensity_scale = utils_for_plotly_rewrites.final_scale(utils_for_plotly_rewrites.min_bright, int_factor)

    color_rgb_float = utils_for_plotly_rewrites.colored_rgb(base_rgb_val, final_intensity_scale)
    # Convert float RGB (0-1) to string 'rgb(R,G,B)' with integer values (0-255)
    color_str = f"rgb({int(color_rgb_float[0]*255)}, {int(color_rgb_float[1]*255)}, {int(color_rgb_float[2]*255)})"

    fig.add_trace(go.Scatter(
        x=[wavelength_start, wavelength_end],
        y=[df_filtered.iloc[i]['Grey Val'], df_filtered.iloc[i+1]['Grey Val']],
        mode='lines',
        line=dict(color=color_str, width=2), # linewidth=2 from line_plot_iteration
        showlegend=False # Don't show legend for individual segments
    ))

# Apply layout settings from axis_labels and other constants
# Calculate width and height based on fig_size = (15,6) and an approximate 80 pixels/inch
calculated_width = fig_size[0] * 60
calculated_height = fig_size[1] * 80

fig.update_layout(
    plot_bgcolor=BG, # 'black' from BG constant
    paper_bgcolor=BG, # 'black' from BG constant
    font=dict(color=COLOUR), # 'white' from COLOUR constant for general text
    title='Grey Val vs. nm (nm range 400-750) with Sharp Peak Labels', # Specific title from original Matplotlib
    width=calculated_width,
    height=calculated_height,
    xaxis=dict(
        title=X_TITLE, # 'Wavelength (nm)' from X_TITLE constant
        range=[X_MIN, X_MAX], # [400, 750] from X_MIN, X_MAX constants
        showgrid=show_grid, # True from show_grid constant
        gridcolor='#333333', # Darker grey for grid, consistent with previous user request
        gridwidth=0.25, # From matplotlib linewidth in axis_labels
        griddash='dot', # Adding dotted grid lines for consistency
        showline=True, # For border box
        linecolor='#333333', # Darker grey for axis line
        linewidth=0.3, # From matplotlib linewidth for spines
        mirror=True, # To create a border box
        tickfont=dict(color=COLOUR), # 'white'
        dtick=50 # From major_locator = ticker.MultipleLocator(50)
    ),
    yaxis=dict(
        title=Y_TITLE, # 'Intensity' from Y_TITLE constant
        showgrid=show_grid, # True
        gridcolor='#333333', # Darker grey for grid
        gridwidth=0.25, # From matplotlib linewidth
        griddash='dot', # Adding dotted grid lines
        showline=True, # For border box
        linecolor='#333333', # Darker grey for axis line
        linewidth=0.3, # From matplotlib linewidth for spines
        mirror=True, # To create a border box
        tickfont=dict(color=COLOUR), # 'white'
        dtick=50 # From major_locator = ticker.MultipleLocator(50)
    )
)

# If show_grid is False, ensure grids are disabled
if not show_grid:
    fig.update_xaxes(showgrid=False)
    fig.update_yaxes(showgrid=False)

fig.show()

In [81]:
import plotly.graph_objects as go

# Create a Plotly figure for the filled plot
fig_filled = go.Figure()

# Iterate through df_filtered to draw line segments with custom colors and fill
for i in range(len(df_filtered) - 1):
    wavelength_start = df_filtered.iloc[i]['nm']
    wavelength_end = df_filtered.iloc[i+1]['nm']

    y_val_start = df_filtered.iloc[i]['Grey Val']
    y_val_end = df_filtered.iloc[i+1]['Grey Val']

    # Calculate color based on wavelength_start (as in Matplotlib version)
    # rgb, gamma_factor are expected to be defined in the global scope.
    base_rgb_tuple = utils_for_plotly_rewrites.rgb(wavelength_start, gamma=gamma_factor) # rgb returns (R,G,B) 0-1

    # Calculate alpha for fill
    # min_alpha, Normalized_int, final_scale are expected to be defined in the global scope.
    alpha_factor = df_filtered.iloc[i]['Normalized_int']
    segment_alpha = utils_for_plotly_rewrites.final_scale(min_alpha, alpha_factor) # min_alpha is 0.1

    # Convert base_rgb_tuple to rgba string for fillcolor
    r, g, b = int(base_rgb_tuple[0]*255), int(base_rgb_tuple[1]*255), int(base_rgb_tuple[2]*255)
    fill_color_str = f"rgba({r}, {g}, {b}, {segment_alpha})"

    # Convert base_rgb_tuple to rgb string for line color
    line_color_str = f"rgb({r}, {g}, {b})"

    fig_filled.add_trace(go.Scatter(
        x=[wavelength_start, wavelength_end],
        y=[y_val_start, y_val_end],
        mode='lines',
        line=dict(color=line_color_str, width=2), # linewidth=2 from line_plot_iteration
        fill='tozeroy', # Fill area below the line to y=0
        fillcolor=fill_color_str,
        showlegend=False,
        name=f'Segment {i}' # Add a name for potential debugging, though not shown
    ))

# Apply layout settings similar to previous Plotly charts
# fig_size, BG, COLOUR, X_TITLE, X_MIN, X_MAX, Y_TITLE, show_grid are global constants
calculated_width = fig_size[0] * 60
calculated_height = fig_size[1] * 80

fig_filled.update_layout(
    plot_bgcolor=BG,
    paper_bgcolor=BG,
    font=dict(color=COLOUR),
    title='Grey Val vs. nm (Filled Plot)', # Custom title for this plot
    width=calculated_width,
    height=calculated_height,
    xaxis=dict(
        title=X_TITLE,
        range=[X_MIN, X_MAX],
        showgrid=show_grid,
        gridcolor='#333333',
        gridwidth=0.25,
        griddash='dot',
        showline=True,
        linecolor='#333333',
        linewidth=0.3,
        mirror=True,
        tickfont=dict(color=COLOUR),
        dtick=50 # From major_locator
    ),
    yaxis=dict(
        title=Y_TITLE,
        showgrid=show_grid,
        gridcolor='#333333',
        gridwidth=0.25,
        griddash='dot',
        showline=True,
        linecolor='#333333',
        linewidth=0.3,
        mirror=True,
        tickfont=dict(color=COLOUR),
        dtick=50 # From major_locator
    )
)

# If show_grid is False, ensure grids are disabled
if not show_grid:
    fig_filled.update_xaxes(showgrid=False)
    fig_filled.update_yaxes(showgrid=False)

fig_filled.show()

In [84]:
import plotly.graph_objects as go
from scipy.signal import find_peaks
import numpy as np # Ensure numpy is imported

def gaussian_iteration_plotly():
    df_plot_data = df_filtered[(df_filtered['nm'] >= 400) & (df_filtered['nm'] <= 750)].copy()
    df_plot_data = df_plot_data.sort_values(by='nm').reset_index(drop=True)

    # Detect peaks
    dyn_prominence = utils_for_plotly_rewrites.dynamic_prominence(0.08, int_range)
    peaks_indices, properties = find_peaks(df_plot_data['Grey Val'], prominence=dyn_prominence)

    # Create a new, denser wavelength array for plotting the synthetic spectrum
    x_synthetic = np.linspace(400, 750, 1000) # 1000 points for a smooth synthetic curve
    y_synthetic = np.zeros_like(x_synthetic)

    for i, peak_idx in enumerate(peaks_indices):
        peak_nm = df_plot_data.iloc[peak_idx]['nm']
        peak_amplitude = df_plot_data.iloc[peak_idx]['Grey Val']
        normalized_amplitude = df_plot_data.iloc[peak_idx]['Normalized_int']

        # Scale sigma based on normalized intensity (higher intensity = broader peak)
        sigma = base_sigma_nm + (max_sigma_multiplier - 1) * base_sigma_nm * normalized_amplitude

        # Create a Gaussian curve for this peak
        gaussian_curve = peak_amplitude * np.exp(-((x_synthetic - peak_nm)**2) / (2 * sigma**2))
        y_synthetic += gaussian_curve # Add to the total synthetic spectrum

    # Normalize the synthetic spectrum intensities for coloring
    min_y_synthetic = y_synthetic.min()
    max_y_synthetic = y_synthetic.max()
    if (max_y_synthetic - min_y_synthetic) == 0:
        normalized_y_synthetic = np.ones_like(y_synthetic)
    else:
        normalized_y_synthetic = (y_synthetic - min_y_synthetic) / (max_y_synthetic - min_y_synthetic)

    # Create Plotly figure
    fig_gaussian = go.Figure()

    # Iterate through each segment of the synthetic spectrum to apply color and alpha
    for i in range(len(x_synthetic) - 1):
        wavelength_start = x_synthetic[i]
        wavelength_end = x_synthetic[i+1]

        y_val_start = y_synthetic[i]
        y_val_end = y_synthetic[i+1]

        base_rgb_tuple = utils_for_plotly_rewrites.rgb(wavelength_start, gamma=gamma_factor)

        segment_alpha = utils_for_plotly_rewrites.final_scale(min_alpha, normalized_y_synthetic[i])

        r, g, b = int(base_rgb_tuple[0]*255), int(base_rgb_tuple[1]*255), int(base_rgb_tuple[2]*255)
        fill_color_str = f"rgba({r}, {g}, {b}, {segment_alpha})"
        # Changed line_color_str to a uniform subtle grey and width to 0 to make the filled area seamless
        line_color_str = '#333333' # Using the subtle grid color for consistency

        fig_gaussian.add_trace(go.Scatter(
            x=[wavelength_start, wavelength_end],
            y=[y_val_start, y_val_end],
            mode='lines',
            line=dict(color=line_color_str, width=0), # Set width to 0 to remove segment outlines
            fill='tozeroy',
            fillcolor=fill_color_str,
            showlegend=False,
            name=f'Gaussian Segment {i}'
        ))

    # Apply layout settings (reusing existing constants)
    calculated_width = fig_size[0] * 80
    calculated_height = fig_size[1] * 80

    fig_gaussian.update_layout(
        plot_bgcolor=BG,
        paper_bgcolor=BG,
        font=dict(color=COLOUR),
        title='Gaussian Synthetic Spectrum (Filled Plot)', # Custom title
        width=calculated_width,
        height=calculated_height,
        xaxis=dict(
            title=X_TITLE,
            range=[X_MIN, X_MAX],
            showgrid=show_grid,
            gridcolor='#1A1A1A',
            gridwidth=0.1,
            griddash='dot',
            showline=True,
            linecolor='#1A1A1A',
            linewidth=0.2,
            mirror=True,
            tickfont=dict(color=COLOUR),
            dtick=50
        ),
        yaxis=dict(
            title=Y_TITLE,
            showgrid=show_grid,
            gridcolor='#1A1A1A',
            gridwidth=0.1,
            griddash='dot',
            showline=True,
            linecolor='#1A1A1A',
            linewidth=0.2,
            mirror=True,
            tickfont=dict(color=COLOUR),
            dtick=50
        )
    )

    if not show_grid:
        fig_gaussian.update_xaxes(showgrid=False)
        fig_gaussian.update_yaxes(showgrid=False)

    fig_gaussian.show()

gaussian_iteration_plotly()

In [85]:
import plotly.graph_objects as go
import numpy as np

def scatter_iteration_plotly():
    alpha_factor = df_filtered['Normalized_int']
    # Adjusted marker sizes to better match Matplotlib's 's' (area) scaling
    # Matplotlib 's' was 5 to 100. Plotly 'size' (diameter) should scale roughly with sqrt(s)
    # sqrt(5) approx 2.23, sqrt(100) = 10.
    # So, we aim for a diameter range of roughly 2 to 10.
    base_marker_size_plotly = 2
    max_marker_size_factor_plotly = 8 # Makes the max size 2 + 8 = 10
    sizes = base_marker_size_plotly + (max_marker_size_factor_plotly * alpha_factor)
    alphas = utils_for_plotly_rewrites.final_scale(min_alpha_scatter, alpha_factor)

    # Prepare a list of RGBA color strings for Plotly
    rgba_colors = []
    for i in range(len(df_filtered)):
        wavelength = df_filtered.iloc[i]['nm']
        r, g, b = utils_for_plotly_rewrites.rgb(wavelength, gamma=gamma_factor) # Ensure gamma_factor is used if rgb function supports it
        a = alphas.iloc[i]
        rgba_colors.append(f"rgba({int(r*255)}, {int(g*255)}, {int(b*255)}, {a})")

    fig_scatter = go.Figure()

    fig_scatter.add_trace(go.Scatter(
        x=df_filtered['nm'],
        y=df_filtered['Grey Val'],
        mode='markers',
        marker=dict(
            color=rgba_colors,
            size=sizes,
            line=dict(
                width=0 # No edge colors for markers
            )
        ),
        showlegend=False,
        name='Emission Data' # Label for the legend if shown, though showlegend is False
    ))

    # Apply layout settings (reusing existing constants)
    calculated_width = fig_size[0] * 80
    calculated_height = fig_size[1] * 80

    fig_scatter.update_layout(
        plot_bgcolor=BG,
        paper_bgcolor=BG,
        font=dict(color=COLOUR),
        title='Emission Data Scatter Plot (Plotly)', # Custom title
        width=calculated_width,
        height=calculated_height,
        xaxis=dict(
            title=X_TITLE,
            range=[X_MIN, X_MAX],
            showgrid=show_grid,
            gridcolor='#1A1A1A',
            gridwidth=0.1,
            griddash='dot',
            showline=True,
            linecolor='#1A1A1A',
            linewidth=0.2,
            mirror=True,
            tickfont=dict(color=COLOUR),
            dtick=50
        ),
        yaxis=dict(
            title=Y_TITLE,
            showgrid=show_grid,
            gridcolor='#1A1A1A',
            gridwidth=0.1,
            griddash='dot',
            showline=True,
            linecolor='#1A1A1A',
            linewidth=0.2,
            mirror=True,
            tickfont=dict(color=COLOUR),
            dtick=50
        )
    )

    if not show_grid:
        fig_scatter.update_xaxes(showgrid=False)
        fig_scatter.update_yaxes(showgrid=False)

    fig_scatter.show()

scatter_iteration_plotly()

In [ ]:
def axis_labels (
    show_grid=True,
    bg_colour = BG,
    text_colour=COLOUR,
    fig_width = 15*60,
    fig_height=6*80,
    x_title=X_TITLE,
    y_title=Y_TITLE,
    x_min=X_MIN,
    x_max=X_MAX,
    grid_colour='#333333',
    major_locator=50,
    minor_locator=10,
    title=None,
    random_title=None,
):
    fig = go.Figure()

    fig.update_layout(
        plot_bgcolor=bg_colour, # 'black' from BG constant
        paper_bgcolor=bg_colour, # 'black' from BG constant
        font=dict(color=text_colour), # 'white' from COLOUR constant for general text
        #title='Grey Val vs. nm (nm range 400-750) with Sharp Peak Labels', # Specific title from original Matplotlib
        width=fig_width,
        height=fig_height,
        xaxis=dict(
            title=x_title, # 'Wavelength (nm)' from X_TITLE constant
            range=[x_min, x_max], # [400, 750] from X_MIN, X_MAX constants
            showgrid=show_grid, # True from show_grid constant
            gridcolor=grid_colour, # Darker grey for grid, consistent with previous user request
            gridwidth=0.25, # From matplotlib linewidth in axis_labels
            griddash='dot', # Adding dotted grid lines for consistency
            showline=True, # For border box
            linecolor=grid_colour, # Darker grey for axis line
            linewidth=0.3, # From matplotlib linewidth for spines
            mirror=True, # To create a border box
            tickfont=dict(color=text_colour), # 'white'
            dtick=major_locator # From major_locator = ticker.MultipleLocator(50)
        ),
        yaxis=dict(
            title=y_title, # 'Intensity' from Y_TITLE constant
            showgrid=show_grid, # True
            gridcolor=grid_colour, # Darker grey for grid
            gridwidth=0.25, # From matplotlib linewidth
            griddash='dot', # Adding dotted grid lines
            showline=True, # For border box
            linecolor=grid_colour, # Darker grey for axis line
            linewidth=0.3, # From matplotlib linewidth for spines
            mirror=True, # To create a border box
            tickfont=dict(color=text_colour), # 'white'
            dtick=major_locator # From major_locator = ticker.MultipleLocator(50)
        )
    )

    # If show_grid is False, ensure grids are disabled
    if not show_grid:
        fig.update_xaxes(showgrid=False)
        fig.update_yaxes(showgrid=False)

In [88]:
def line_iter(df_filtered):
    for i in range(len(df_filtered) - 1):
        wavelength_start = df_filtered.iloc[i]['nm']
        wavelength_end = df_filtered.iloc[i+1]['nm']

        int_factor = df_filtered.iloc[i]['Normalized_int']
        # rgb, colored_rgb, final_scale, gamma_factor, min_bright are expected to be defined in the global scope.
        base_rgb_val = utils_for_plotly_rewrites.rgb(wavelength_start, gamma=gamma_factor)
        final_intensity_scale = utils_for_plotly_rewrites.final_scale(utils_for_plotly_rewrites.min_bright, int_factor)

        color_rgb_float = utils_for_plotly_rewrites.colored_rgb(base_rgb_val, final_intensity_scale)
        # Convert float RGB (0-1) to string 'rgb(R,G,B)' with integer values (0-255)
        color_str = f"rgb({int(color_rgb_float[0]*255)}, {int(color_rgb_float[1]*255)}, {int(color_rgb_float[2]*255)})"

        fig.add_trace(go.Scatter(
            x=[wavelength_start, wavelength_end],
            y=[df_filtered.iloc[i]['Grey Val'], df_filtered.iloc[i+1]['Grey Val']],
            mode='lines',
            line=dict(color=color_str, width=2), # linewidth=2 from line_plot_iteration
            showlegend=False # Don't show legend for individual segments
        ))

In [91]:
axis_labels()
line_iter(df_filtered=df_filtered)
fig.show()

In [92]:
import plotly.graph_objects as go

def bar_iteration_plotly():
    bar_colors = []
    for index, row in df_filtered.iterrows():
        wavelength = row['nm']
        normalized_intensity = row['Normalized_int']

        base_rgb_tuple = utils_for_plotly_rewrites.rgb(wavelength, gamma=gamma_factor);

        # Use a local minimum brightness specifically for the bar plot
        # to ensure vibrancy on a black background, while maintaining intensity scaling
        local_bar_min_bright = 0.3 # Locally set a minimum for bar brightness for Plotly on black
        final_intensity_scale_val = utils_for_plotly_rewrites.final_scale(local_bar_min_bright, normalized_intensity);

        # Manually apply the colored_rgb logic to scale R, G, B components
        r_scaled = base_rgb_tuple[0] * final_intensity_scale_val;
        g_scaled = base_rgb_tuple[1] * final_intensity_scale_val;
        b_scaled = base_rgb_tuple[2] * final_intensity_scale_val;

        # Convert to 0-255 integers
        r, g, b = int(r_scaled * 255), int(g_scaled * 255), int(b_scaled * 255);

        # Use a fixed alpha of 1.0 (fully opaque) as the intensity is now baked into RGB
        bar_colors.append(f"rgba({r}, {g}, {b}, {1.0})");

    fig_bar = go.Figure();

    fig_bar.add_trace(go.Bar(
        x=df_filtered['nm'],
        y=df_filtered['Grey Val'],
        marker=dict(color=bar_colors), # Use marker dict for color
        width=bar_width, # Use the global bar_width for Plotly bar width
        showlegend=False,
        name='Emission Data Bars'
    ));

    # Apply layout settings (reusing existing constants)
    calculated_width = fig_size[0] * 80;
    calculated_height = fig_size[1] * 80;

    fig_bar.update_layout(
        plot_bgcolor=BG,
        paper_bgcolor=BG,
        font=dict(color=COLOUR),
        title='Emission Data Bar Plot (Plotly)', # Custom title
        width=calculated_width,
        height=calculated_height,
        xaxis=dict(
            title=X_TITLE,
            range=[X_MIN, X_MAX],
            showgrid=show_grid,
            gridcolor='#1A1A1A',
            gridwidth=0.1,
            griddash='dot',
            showline=True,
            linecolor='#1A1A1A',
            linewidth=0.2,
            mirror=True,
            tickfont=dict(color=COLOUR),
            dtick=50
        ),
        yaxis=dict(
            title=Y_TITLE,
            showgrid=show_grid,
            gridcolor='#1A1A1A',
            gridwidth=0.1,
            griddash='dot',
            showline=True,
            linecolor='#1A1A1A',
            linewidth=0.2,
            mirror=True,
            tickfont=dict(color=COLOUR),
            dtick=50
        )
    );

    if not show_grid:
        fig_bar.update_xaxes(showgrid=False);
        fig_bar.update_yaxes(showgrid=False);

    fig_bar.show();

bar_iteration_plotly();